# Day 11: Cleaned Company Employee Dataset

This notebook loads the messy employee data, profiles missing and inconsistent values, applies documented cleaning rules, verifies the result, and exports the cleaned dataset and a cleaning summary.

In [1]:
from pathlib import Path
import pandas as pd

source_path = Path(r"c:\Users\HP\Searches\Internship\Day11_Messy_Company_Employee_Dataset.csv")
output_path = source_path.with_name("Day11_Cleaned_Company_Employee_Dataset.csv")
summary_path = source_path.with_name("Day11_Cleaning_Summary.txt")

raw = pd.read_csv(source_path, na_values=["", "N/A", "NA", "null", "None"])
print(f"Rows: {raw.shape[0]}, columns: {raw.shape[1]}")
display(raw.head())
print("\nData types before cleaning:")
print(raw.dtypes)

missing_before = raw.isnull().sum().sort_values(ascending=False)
print("\nMissing values before cleaning:")
display(missing_before[missing_before > 0].to_frame("missing_count"))

Rows: 157, columns: 12


,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
0,EMP0098,Ananya Nair,NaN,Data Scientist,48.0,Other,108371.0,13.2,2017-06-13,Pune,5.0,Office
1,EMP0046,Faizan Khan,Marketing,Marketing Manager,31.0,Female,108824.0,12.1,2021-09-15,Mumbai,2.0,Office
2,EMP0017,Saira Malik,Human Resources,HR Manager,28.0,Female,119400.0,13.2,2021-02-05,Bengaluru,3.0,Remote
3,EMP0105,Neha Menon,Engineering,Software Engineer,30.0,Female,111847.0,NaN,2018-08-05,Jaipur,4.0,Hybrid
4,EMP0143,Hiba Menon,Sales,Sales Manager,25.0,Female,78677.0,10.9,2018-07-24,Hyderabad,4.0,Hybrid



Data types before cleaning:
Employee_ID           object
Employee_Name         object
Department            object
Job_Title             object
Age                  float64
Gender                object
Annual_Salary        float64
Experience_Years     float64
Joining_Date          object
City                  object
Performance_Score    float64
Work_Mode             object
dtype: object

Missing values before cleaning:


,missing_count
Department,5
Gender,5
Annual_Salary,5
City,5
Age,4
Experience_Years,3
Performance_Score,3
Work_Mode,2


In [2]:
text_columns = ["Department", "Gender", "City", "Work_Mode"]
print(f"Exact duplicate rows: {raw.duplicated().sum()}")
print(f"Duplicate Employee_ID rows: {raw.duplicated('Employee_ID').sum()}")

print("\nPotential categorical inconsistencies before cleaning:")
for column in text_columns:
    values = raw[column].dropna().astype(str)
    normalized = values.str.strip().str.casefold()
    inconsistent_count = values[values.str.strip() != values].size + (values.str.casefold() != normalized).sum()
    print(f"{column}: {raw[column].nunique(dropna=True)} raw values; {normalized.nunique()} case/whitespace-normalized values")

print("\nInvalid dates before cleaning:", pd.to_datetime(raw["Joining_Date"], errors="coerce").isna().sum())
print("\nNumeric summary before cleaning:")
display(raw[["Age", "Annual_Salary", "Experience_Years", "Performance_Score"]].describe().T)

Exact duplicate rows: 7
Duplicate Employee_ID rows: 7

Potential categorical inconsistencies before cleaning:
Department: 10 raw values; 8 case/whitespace-normalized values
Gender: 5 raw values; 3 case/whitespace-normalized values
City: 13 raw values; 10 case/whitespace-normalized values
Work_Mode: 5 raw values; 3 case/whitespace-normalized values

Invalid dates before cleaning: 0

Numeric summary before cleaning:


,count,mean,std,min,25%,50%,75%,max
Age,153.0,38.130719,9.361661,21.0,30.00,39.00,46.0,52.0
Annual_Salary,152.0,89240.592105,22626.172309,43331.0,71198.50,90046.50,107328.0,162942.0
Experience_Years,154.0,8.990909,4.983874,0.6,4.45,9.35,13.2,17.4
Performance_Score,154.0,3.525974,0.930233,2.0,3.00,4.00,4.0,5.0


## Cleaning decisions

- Remove exact duplicate rows with `drop_duplicates()`; duplicate employee IDs are exact repeats, so no distinct employee information is lost.
- Treat blank strings and `N/A` as missing values.
- Strip whitespace and standardize known categorical labels (`Engineering`, `Female`, `Delhi`, and `Remote`).
- Use mode imputation for categorical columns and median imputation for numeric columns, because the small number of missing values should not be allowed to distort the data.
- Parse `Joining_Date` as a datetime and drop rows missing required identity fields only.

In [3]:
cleaned = raw.copy()

# Remove repeated complete records before imputing values.
cleaned = cleaned.drop_duplicates().copy()

# Normalize text values and known label variants.
for column in text_columns:
    cleaned[column] = cleaned[column].astype("string").str.strip()

label_maps = {
    "Department": {"ENGINEERING": "Engineering", "engineering": "Engineering"},
    "Gender": {"female": "Female", "FEMALE": "Female", "MALE": "Male"},
    "City": {"delhi": "Delhi", "DELHI": "Delhi", "Delhi ": "Delhi"},
    "Work_Mode": {"REMOTE": "Remote", "remote": "Remote"},
}
for column, mapping in label_maps.items():
    cleaned[column] = cleaned[column].replace(mapping)

# Required fields are identifying fields; discard only records missing them.
cleaned = cleaned.dropna(subset=["Employee_ID", "Employee_Name", "Job_Title", "Joining_Date"])

# Infer missing departments from job titles before using the department mode as fallback.
title_department = {
    "Data Scientist": "Data & Analytics",
    "BI Analyst": "Data & Analytics",
    "Software Engineer": "Engineering",
    "Senior Software Engineer": "Engineering",
    "QA Engineer": "Engineering",
    "Business Development Executive": "Sales",
    "Customer Success Executive": "Customer Success",
}
missing_department = cleaned["Department"].isna()
cleaned.loc[missing_department, "Department"] = cleaned.loc[missing_department, "Job_Title"].map(title_department)

categorical_columns = ["Department", "Gender", "City", "Work_Mode"]
numeric_columns = ["Age", "Annual_Salary", "Experience_Years", "Performance_Score"]
for column in categorical_columns:
    cleaned[column] = cleaned[column].fillna(cleaned[column].mode().iloc[0])
for column in numeric_columns:
    cleaned[column] = cleaned[column].fillna(cleaned[column].median())

cleaned["Joining_Date"] = pd.to_datetime(cleaned["Joining_Date"], errors="coerce")
cleaned = cleaned.dropna(subset=["Joining_Date"])

# Store whole-number fields as integers and keep experience precision.
cleaned["Age"] = cleaned["Age"].round().astype("int64")
cleaned["Annual_Salary"] = cleaned["Annual_Salary"].round().astype("int64")
cleaned["Performance_Score"] = cleaned["Performance_Score"].round().astype("int64")
cleaned["Joining_Date"] = cleaned["Joining_Date"].dt.strftime("%Y-%m-%d")

print(f"Rows after cleaning: {cleaned.shape[0]}, columns: {cleaned.shape[1]}")
display(cleaned.head())

Rows after cleaning: 150, columns: 12


,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
0,EMP0098,Ananya Nair,Data & Analytics,Data Scientist,48,Other,108371,13.2,2017-06-13,Pune,5,Office
1,EMP0046,Faizan Khan,Marketing,Marketing Manager,31,Female,108824,12.1,2021-09-15,Mumbai,2,Office
2,EMP0017,Saira Malik,Human Resources,HR Manager,28,Female,119400,13.2,2021-02-05,Bengaluru,3,Remote
3,EMP0105,Neha Menon,Engineering,Software Engineer,30,Female,111847,9.2,2018-08-05,Jaipur,4,Hybrid
4,EMP0143,Hiba Menon,Sales,Sales Manager,25,Female,78677,10.9,2018-07-24,Hyderabad,4,Hybrid


In [4]:
missing_after = cleaned.isnull().sum()
invalid_dates_after = pd.to_datetime(cleaned["Joining_Date"], errors="coerce").isna().sum()
comparison = pd.DataFrame({
    "Before": [len(raw), raw.duplicated().sum(), int(raw.isnull().sum().sum())],
    "After": [len(cleaned), cleaned.duplicated().sum(), int(cleaned.isnull().sum().sum())],
}, index=["Rows", "Exact duplicate rows", "Missing cells"])

print("Before vs. after cleaning:")
display(comparison)
print("\nMissing values after cleaning:")
display(missing_after.to_frame("missing_count"))
print("Data types after cleaning:")
print(cleaned.dtypes)
print("\nUnique standardized categorical values:")
for column in categorical_columns:
    print(f"{column}: {sorted(cleaned[column].dropna().unique().tolist())}")

assert cleaned.isnull().sum().sum() == 0
assert cleaned.duplicated().sum() == 0
assert cleaned["Employee_ID"].duplicated().sum() == 0
assert invalid_dates_after == 0
assert cleaned["Performance_Score"].between(1, 5).all()
assert cleaned["Age"].between(18, 100).all()

cleaning_summary = f"""Day 11 Cleaning Summary

Source: {source_path.name}
Output: {output_path.name}

Before cleaning:
- Rows: {len(raw)}
- Missing cells: {int(raw.isnull().sum().sum())}
- Exact duplicate rows: {int(raw.duplicated().sum())}
- Duplicate Employee_ID rows: {int(raw.duplicated('Employee_ID').sum())}

Cleaning steps:
- Converted blanks and N/A markers to missing values.
- Removed exact duplicate records with drop_duplicates().
- Trimmed whitespace and standardized categorical labels for department, gender, city, and work mode.
- Inferred missing departments from job titles, then used mode imputation for remaining categorical gaps.
- Used median imputation for missing age, salary, experience, and performance scores.
- Parsed joining dates as datetime values and removed invalid required dates.
- Converted age, salary, and performance score to integer types.

After cleaning:
- Rows: {len(cleaned)}
- Missing cells: {int(cleaned.isnull().sum().sum())}
- Exact duplicate rows: {int(cleaned.duplicated().sum())}
- Duplicate Employee_ID rows: {int(cleaned.duplicated('Employee_ID').sum())}
- Invalid dates: {invalid_dates_after}
"""

cleaned.to_csv(output_path, index=False)
summary_path.write_text(cleaning_summary, encoding="utf-8")
print(f"\nExported cleaned dataset to: {output_path}")
print(f"Exported cleaning summary to: {summary_path}")

Before vs. after cleaning:


,Before,After
Rows,157,150
Exact duplicate rows,7,0
Missing cells,32,0



Missing values after cleaning:


,missing_count
Employee_ID,0
Employee_Name,0
Department,0
Job_Title,0
Age,0
Gender,0
Annual_Salary,0
Experience_Years,0
Joining_Date,0
City,0


Data types after cleaning:
Employee_ID                  object
Employee_Name                object
Department           string[python]
Job_Title                    object
Age                           int64
Gender               string[python]
Annual_Salary                 int64
Experience_Years            float64
Joining_Date                 object
City                 string[python]
Performance_Score             int64
Work_Mode            string[python]
dtype: object

Unique standardized categorical values:
Department: ['Customer Success', 'Data & Analytics', 'Engineering', 'Finance', 'Human Resources', 'Marketing', 'Operations', 'Sales']
Gender: ['Female', 'Male', 'Other']
City: ['Bengaluru', 'Chandigarh', 'Chennai', 'Delhi', 'Hyderabad', 'Jaipur', 'Kolkata', 'Mumbai', 'Pune', 'Srinagar']
Work_Mode: ['Hybrid', 'Office', 'Remote']

Exported cleaned dataset to: c:\Users\HP\Searches\Internship\Day11_Cleaned_Company_Employee_Dataset.csv
Exported cleaning summary to: c:\Users\HP\Searches\